# GFF Parsing with Biopython (BCBio)

This notebook demonstrates how to parse a GFF3 file using `bcbio-gff` as suggested in the [Biopython Wiki](https://biopython.org/wiki/GFF_Parsing).

In [7]:
from BCBio import GFF
from Bio import SeqIO
import os
import pandas as pd
import os
import glob
import subprocess
import shutil
import gc

# MOL_TYPE

In [9]:


# 1. Define Paths
data_dir = '../viral_data_all/ncbi_dataset/data_subset/'

# 2. Identify the target FASTA files
# finding all files starting with GCA, GCF, ending with _genomic.fna, without 'cds'
file_pattern = '**/genomic.gff'

gff_files = [f for f in glob.glob(os.path.join(data_dir, file_pattern), recursive=True) if 'cds' not in os.path.basename(f)]

print(f"Found {len(gff_files)} GFF files matching the pattern.")

Found 211307 GFF files matching the pattern.


In [8]:
# --- Function: extract mol_type from a GFF file ---
def extract_mol_type(gff_file):
    """Extract mol_type from region features in a GFF file.
    Returns a list of dicts with record_id, mol_type, and accession."""
    
    # Extract accession folder name from path
    # e.g., "../viral_data_all/ncbi_dataset/data_subset/GCF_000864765.1/genomic.gff"
    #  → "GCF_000864765.1"
    accession = os.path.basename(os.path.dirname(gff_file))
    
    results = []
    try:
        with open(gff_file) as in_handle:
            for record in GFF.parse(in_handle):
                for feature in record.features:
                    if feature.type == "region":
                        mol_type = feature.qualifiers.get("mol_type", [None])[0]
                        results.append({
                            "record_id": record.id,
                            "accession": accession,
                            "mol_type": mol_type,
                            "source_file": gff_file
                        })
    except Exception as e:
        print(f"Error parsing {gff_file}: {e}")
        # Still try raw parsing as fallback
        try:
            results.extend(extract_mol_type_raw(gff_file))
        except Exception as e2:
            print(f"  Raw fallback also failed: {e2}")
    
    return results


    
def extract_mol_type_raw(gff_file):
    """Fallback: parse GFF as plain text to extract mol_type from region lines."""
    accession = os.path.basename(os.path.dirname(gff_file))
    results = []
    with open(gff_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 9 and parts[2] == "region":
                attrs = parts[8]
                record_id = parts[0]
                mol_type = None
                for attr in attrs.split(";"):
                    if attr.startswith("mol_type="):
                        mol_type = attr.split("=", 1)[1]
                        break
                results.append({
                    "record_id": record_id,
                    "accession": accession,
                    "mol_type": mol_type,
                    "source_file": gff_file
                })
    return results

In [10]:
all_matches = []
## Error files already handled within the original function ####
checkpoint_dir = "../gff_results/mol_type_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_interval = 10000
checkpoint_count = 0
for i, gff_file in enumerate(gff_files):
    matches = extract_mol_type(gff_file)
    all_matches.extend(matches)
    
    # Checkpoint: save and flush every 10,000 files
    if (i + 1) % checkpoint_interval == 0:
        if all_matches:
            df_chunk = pd.DataFrame(all_matches)
            df_chunk.to_csv(f"{checkpoint_dir}/checkpoint_{checkpoint_count}.csv", index=False)
            print(f"[{i+1}/{len(gff_files)}] Saved checkpoint_{checkpoint_count}.csv ({len(df_chunk)} rows)")
            checkpoint_count += 1
            all_matches = []  # free memory
            del df_chunk
            gc.collect()
        else:
            print(f"[{i+1}/{len(gff_files)}] No matches in this batch, skipping checkpoint.")
# Save any remaining matches after the loop
if all_matches:
    df_chunk = pd.DataFrame(all_matches)
    df_chunk.to_csv(f"{checkpoint_dir}/checkpoint_{checkpoint_count}.csv", index=False)
    print(f"[{len(gff_files)}/{len(gff_files)}] Saved final checkpoint_{checkpoint_count}.csv ({len(df_chunk)} rows)")
    del df_chunk
    gc.collect()
# # --- Merge all checkpoints into one DataFrame ---
# checkpoint_files = sorted(
#     [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".csv")]
# )
# print(f"\nMerging {len(checkpoint_files)} checkpoint files...")
# df_mol = pd.concat([pd.read_csv(f"{checkpoint_dir}/{f}") for f in checkpoint_files], ignore_index=True)
# df_mol = df_mol.drop_duplicates(subset=["record_id", "accession", "mol_type"])
# print(f"Found {len(df_mol)} unique records with mol_type across {len(gff_files)} files.")
# print(f"\nmol_type value counts:")
# print(df_mol["mol_type"].value_counts())
# display(df_mol.head(20))

Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCA_003108865.1/genomic.gff: Did not find remapped ID location: gene-tat, [[5846, 6061], [8371, 8462]], [5846, 8462]
[10000/211307] Saved checkpoint_0.csv (63094 rows)
Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCA_003108965.1/genomic.gff: Did not find remapped ID location: gene-tat, [[5895, 6107], [8461, 8600]], [5895, 8600]
[20000/211307] Saved checkpoint_1.csv (62925 rows)
Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCA_031239455.1/genomic.gff: Did not find remapped ID location: gene-tRNA-Arg (TCT), [[101529, 101568], [101482, 101519]], [101482, 101568]
[30000/211307] Saved checkpoint_2.csv (63002 rows)
[40000/211307] Saved checkpoint_3.csv (62977 rows)
[50000/211307] Saved checkpoint_4.csv (62713 rows)
[60000/211307] Saved checkpoint_5.csv (63142 rows)
Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCF_000844405.1/genomic.gff: Did not find remapped ID location: gene-CcBV_6.3, [[14873, 1545

In [13]:

checkpoint_dir = "../gff_results/mol_type_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- Merge all checkpoints into one DataFrame ---
checkpoint_files = sorted(
    [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".csv")]
)
print(f"\nMerging {len(checkpoint_files)} checkpoint files from {checkpoint_dir}...")

df_all = pd.concat([pd.read_csv(f"{checkpoint_dir}/{f}") for f in checkpoint_files], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["record_id", "accession", "mol_type", "source_file"])
print(f"Found {len(df_all)} unique features.")
display(df_all.drop(columns=['qualifiers'], errors='ignore').head(20))

df_all.to_csv("../gff_results/mol_type_parsed.csv", index=False)
print("Saved combined df to mol_type_parsed.csv")


Merging 22 checkpoint files from ../gff_results/mol_type_checkpoints...
Found 1327153 unique features.


,record_id,accession,mol_type,source_file
0,MG982672.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,MG982710.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,MG982782.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,MG982795.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,MG982796.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
5,MG982799.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
6,MG982833.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
7,MG982864.1,GCA_038807845.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
8,MT090509.1,GCA_039170005.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...
9,MT090510.1,GCA_039170005.1,viral cRNA,../viral_data_all/ncbi_dataset/data_subset/GCA...


Saved combined df to mol_type_parsed.csv


# CHECKING


In [14]:
df = pd.read_csv("../gff_results/mol_type_parsed.csv")

In [ ]:
df.head()


df.groupby('accession')['mol_type'].nunique().loc[lambda x: x > 1]
df.groupby('accession')['mol_type']



accession
GCA_000820495.2    2
GCA_000854145.2    2
GCA_000854165.1    2
GCA_000854765.1    2
GCA_000854765.2    2
                  ..
GCF_001755065.1    2
GCF_001755105.1    2
GCF_001755485.1    2
GCF_001755805.1    2
GCF_002833885.1    2
Name: mol_type, Length: 69, dtype: int64

In [ ]:
from BCBio import GFF
gff_file = "../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff"
with open(gff_file) as in_handle:
    for record in GFF.parse(in_handle):
        print(f"Record: {record.id}")
        for feature in record.features:
            print(f"  Feature: {feature.type}, {feature.location}")
            # To see mol_type if it exists in the top-level record:
            # print(record.annotations.get('molecule_type'))

In [25]:
import pandas as pd
gff_file = "../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff"
# GFF files are tab-separated; we skip lines starting with '#'
df_gff = pd.read_csv(
    gff_file, 
    sep='\t', 
    comment='#', 
    header=None,
    names=["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
)
# To see the mol_type, it's usually inside the 'attributes' column for 'region' type features
print(df_gff[df_gff['type'] == 'region']['attributes'].iloc[0])

ID=M14880.1:1..2368;Dbxref=taxon:518987;Name=1;clone=pBP4;gbkey=Src;mol_type=genomic RNA;old-name=Influenza B virus (B/Lee/40);segment=1;strain=B/Lee/40


In [26]:
extract_mol_type(gff_file)

[{'record_id': 'AF101982.1',
  'accession': 'GCA_000820495.2',
  'mol_type': 'genomic RNA',
  'source_file': '../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff'},
 {'record_id': 'AF102017.1',
  'accession': 'GCA_000820495.2',
  'mol_type': 'genomic RNA',
  'source_file': '../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff'},
 {'record_id': 'J02094.1',
  'accession': 'GCA_000820495.2',
  'mol_type': 'viral cRNA',
  'source_file': '../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff'},
 {'record_id': 'J02095.1',
  'accession': 'GCA_000820495.2',
  'mol_type': 'viral cRNA',
  'source_file': '../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff'},
 {'record_id': 'J02096.1',
  'accession': 'GCA_000820495.2',
  'mol_type': 'viral cRNA',
  'source_file': '../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff'},
 {'record_id': 'K00423.1',
  'accession': 'GCA_000820495.2',
  'mol_type': 'viral cRNA',
